# EDAT vs no-EDAT (F1 Comparison)

This notebook compares:
- Baseline run (no EDAT)
- Best EDAT run

It reports absolute and relative F1 improvement (delta).



In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Iterable

# Optional dependency for reading torch checkpoints.
try:
    import torch
except Exception:
    torch = None

ROOT = Path.cwd()
print(f"Working directory: {ROOT}")

Working directory: /content


## 1) Configure Input Mode
Choose one mode:
- `manual`: You type baseline and EDAT F1 directly.
- `auto`: Notebook scans files and tries to extract F1 values.

In [2]:
# --- USER CONFIG ---
MODE = "manual"  # "manual" or "auto"

# Manual mode values
BASELINE_F1_MANUAL = 0.70
EDAT_BEST_F1_MANUAL = 0.75

# Auto mode search settings
SEARCH_ROOT = ROOT
MAX_FILES_TO_SCAN = 300

# If your file names include these words, detection is easier.
BASELINE_HINTS = ["baseline", "no_edat", "no-edat", "without_edat"]
EDAT_HINTS = ["edat"]

## 2) Helper Functions
These helpers parse F1 from JSON/TXT/LOG files and optional `.pt/.pth/.ckpt` files.

In [3]:
F1_KEY_PATTERNS = [
    re.compile(r"^f1$", re.IGNORECASE),
    re.compile(r"val[_\- ]?f1", re.IGNORECASE),
    re.compile(r"macro[_\- ]?f1", re.IGNORECASE),
]

F1_IN_TEXT_PATTERN = re.compile(r"(?:val[_\- ]?f1|macro[_\- ]?f1|f1)\s*[:=]\s*([0-9]*\.?[0-9]+)", re.IGNORECASE)

def _looks_like_f1_key(key: str) -> bool:
    key = str(key)
    return any(p.search(key) for p in F1_KEY_PATTERNS)

def _collect_f1_from_obj(obj: Any, out: list[float]) -> None:
    if isinstance(obj, dict):
        for k, v in obj.items():
            if _looks_like_f1_key(k) and isinstance(v, (int, float)):
                val = float(v)
                if 0.0 <= val <= 1.0:
                    out.append(val)
            _collect_f1_from_obj(v, out)
    elif isinstance(obj, list):
        for item in obj:
            _collect_f1_from_obj(item, out)

def _extract_f1_from_text(text: str) -> list[float]:
    vals = []
    for m in F1_IN_TEXT_PATTERN.finditer(text):
        val = float(m.group(1))
        if 0.0 <= val <= 1.0:
            vals.append(val)
    return vals

def extract_f1_candidates(path: Path) -> list[float]:
    suffix = path.suffix.lower()
    vals: list[float] = []

    if suffix in {".json", ".jsonl"}:
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
            if suffix == ".json":
                obj = json.loads(text)
                _collect_f1_from_obj(obj, vals)
            else:
                for line in text.splitlines():
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                        _collect_f1_from_obj(obj, vals)
                    except Exception:
                        vals.extend(_extract_f1_from_text(line))
        except Exception:
            pass

    elif suffix in {".txt", ".log", ".md", ".yaml", ".yml"}:
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
            vals.extend(_extract_f1_from_text(text))
        except Exception:
            pass

    elif suffix in {".pt", ".pth", ".ckpt"} and torch is not None:
        try:
            obj = torch.load(path, map_location="cpu")
            _collect_f1_from_obj(obj, vals)
        except Exception:
            pass

    return vals

def has_any_hint(path: Path, hints: Iterable[str]) -> bool:
    name = str(path).lower()
    return any(h.lower() in name for h in hints)

## 3) Resolve Baseline and EDAT Best F1
In `auto` mode, this scans the project for likely files and extracts max F1 from each side.

In [4]:
def auto_resolve_f1(search_root: Path):
    patterns = ["*.json", "*.jsonl", "*.txt", "*.log", "*.md", "*.yaml", "*.yml", "*.pt", "*.pth", "*.ckpt"]
    files = []
    for p in patterns:
        files.extend(search_root.rglob(p))

    files = files[:MAX_FILES_TO_SCAN]

    baseline_vals = []
    edat_vals = []

    for fp in files:
        vals = extract_f1_candidates(fp)
        if not vals:
            continue

        if has_any_hint(fp, BASELINE_HINTS):
            baseline_vals.extend(vals)

        if has_any_hint(fp, EDAT_HINTS):
            edat_vals.extend(vals)

    return baseline_vals, edat_vals

if MODE == "manual":
    baseline_f1 = float(BASELINE_F1_MANUAL)
    edat_best_f1 = float(EDAT_BEST_F1_MANUAL)
    scan_info = "Manual mode: used user-provided values."

elif MODE == "auto":
    b_vals, e_vals = auto_resolve_f1(SEARCH_ROOT)
    if not b_vals:
        raise ValueError("Could not find baseline F1 automatically. Add hints or switch to manual mode.")
    if not e_vals:
        raise ValueError("Could not find EDAT F1 automatically. Add hints or switch to manual mode.")

    baseline_f1 = max(b_vals)
    edat_best_f1 = max(e_vals)
    scan_info = f"Auto mode: baseline candidates={len(b_vals)}, EDAT candidates={len(e_vals)}"

else:
    raise ValueError("MODE must be 'manual' or 'auto'.")

print(scan_info)
print(f"baseline_f1 = {baseline_f1:.6f}")
print(f"edat_best_f1 = {edat_best_f1:.6f}")

Manual mode: used user-provided values.
baseline_f1 = 0.700000
edat_best_f1 = 0.750000


## 4) Compute Improvement Delta

In [5]:
delta_abs = edat_best_f1 - baseline_f1
delta_rel_pct = (delta_abs / baseline_f1 * 100.0) if baseline_f1 != 0 else float("inf")

direction = "improvement" if delta_abs >= 0 else "drop"

print("=== EDAT Comparison Report ===")
print(f"Baseline (no EDAT) F1 : {baseline_f1:.4f}")
print(f"Best EDAT run F1      : {edat_best_f1:.4f}")
print(f"Absolute delta         : {delta_abs:+.4f}")
print(f"Relative delta         : {delta_rel_pct:+.2f}%")
print(f"Result                 : {direction}")

=== EDAT Comparison Report ===
Baseline (no EDAT) F1 : 0.7000
Best EDAT run F1      : 0.7500
Absolute delta         : +0.0500
Relative delta         : +7.14%
Result                 : improvement


## 5) Optional: One-line Summary for Report

In [6]:
summary = (
    f"Baseline (no EDAT) F1 = {baseline_f1:.4f}; "
    f"Best EDAT F1 = {edat_best_f1:.4f}; "
    f"Delta = {delta_abs:+.4f} ({delta_rel_pct:+.2f}%)."
)
print(summary)

Baseline (no EDAT) F1 = 0.7000; Best EDAT F1 = 0.7500; Delta = +0.0500 (+7.14%).
